In [0]:
import dlt
from pyspark.sql.functions import *

dlt.create_streaming_table("electroniz_silver_store_orders")

In [0]:
@dlt.view
def v_latest_currency_rates():
    df = spark.read.table("electroniz_catalog.electroniz_silver_schema.electroniz_silver_currency")
    return df.orderBy(col("date").desc()).select("EUR", "USD", "CAD", "INR").limit(1)

In [0]:
@dlt.view
@dlt.expect_or_drop("valid_order_mode", "order_mode IN ('NEW', 'DELETE', 'EDIT')")
@dlt.expect_or_drop("valid_currency", "currency IN ('USD', 'INR', 'EUR', 'CAD')")
def v_clean_store_orders():
    bronze = spark.readStream.table("electroniz_catalog.electroniz_bronze_schema.electroniz_bronze_store_orders")
    rates = dlt.read("v_latest_currency_rates")
    return bronze.crossJoin(rates).select(
        col("order_number"),
        col("customer_id"),
        col("product_id"),
        coalesce(
            to_date(col("order_date"), "dd/MM/yyyy"),
            to_date(col("order_date"), "yyyy-MM-dd"),
            to_date(col("order_date"), "MM/dd/yyyy")
        ).alias("order_date"),
        col("units"),
        col("sale_price").cast("double").alias("sale_price"),
        col("currency"),
        col("order_mode"),
        col("updated_at"),
        (col("sale_price").cast("double") /
         when(col("currency") == "EUR", col("EUR"))
         .when(col("currency") == "USD", col("USD"))
         .when(col("currency") == "CAD", col("CAD"))
         .when(col("currency") == "INR", col("INR"))
        ).cast("float").alias("sale_price_eur")
    )

In [0]:
dlt.apply_changes(
    target="electroniz_silver_store_orders",
    source="v_clean_store_orders",
    keys=["order_number", "customer_id"],
    sequence_by=col("updated_at"),
    stored_as_scd_type=2
)